# ETL Validation: Population Fidelity & Analytic Fidelity

Two quantitative validation efforts for the A4/LEARN OMOP CDM v5.4 ETL, comparing
cohorts and analyses derived **independently** from the source files (`Raw Data/`,
`Derived Data/`, `External Data/`) and from the OMOP output (`OMOP_Output/` CSVs).

**Effort 1 — Population / cohort fidelity.** For a battery of cohort definitions,
each derived twice (once from source, once from OMOP), we report participant-level
overlap as the Jaccard similarity `|S ∩ O| / |S ∪ O|` on blinded subject IDs
(`BID` ↔ `person_source_value`), plus asymmetric discordance counts.

**Effort 2 — Analytic fidelity.** The same analytical pipeline is run on each
representation's *native* values and the performance deviation
`Δ = Performance_OMOP − Performance_Source` is estimated with a **paired design**:
the comparison is restricted to subjects present on both sides, every subject is
assigned to the same cross-validation fold on both sides (deterministic MD5 hash of
BID), and the Δ confidence interval comes from a paired bootstrap over subjects.
Two tasks mirror the existing analysis notebook:

- **Task A** (mirrors Analysis 2): plasma p-tau217 predicting amyloid-PET
  positivity (screening composite ≥ 20 centiloids), single-feature AUROC.
- **Task B** (mirrors Analysis 5): multi-modal prediction of cognitive decline
  (PACC change from baseline < −1 SD at last follow-up) with the identical
   18-feature set, models (LR / RF / GB, fixed seeds), and imputation.

**Parity policy: measure the transforms.** The ETL's intentional transforms
(unit conversions such as SMOKE packs → cigarettes ×20 and Roche Abeta ng → pg,
`<LLOQ` raw-value recovery, QC-row and sentinel filtering, year-precision ages)
are *not* harmonized away — each side uses its native values so the validation
measures exactly what a downstream analyst would experience. Cohort criteria are
expressed natively per side (e.g. "≥1 pack/day" vs "≥20 cigarettes/day").

Note: one Analysis 5 feature pattern is corrected here — MMSE totals moved from
MEASUREMENT to OBSERVATION on 2026-08-21 (domain routing), so this notebook reads
concept 42869860 from `observation.csv`.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import hashlib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, cohen_kappa_score

warnings.filterwarnings('ignore')
rng = np.random.default_rng(42)

BASE = Path('..') if Path.cwd().name == 'analyses' else Path('.')
OMOP = BASE / 'OMOP_Output'
RESULTS_DIR = BASE / 'analyses' / 'results'

C_OMOP, C_SOURCE = '#0072B2', '#E69F00'   # CVD-safe pair, validated

# OMOP tables (needed columns only)
person = pd.read_csv(OMOP / 'person.csv',
                     usecols=['person_id', 'person_source_value', 'year_of_birth',
                              'gender_concept_id', 'race_concept_id'])
PID2BID = person.set_index('person_id')['person_source_value']
obs = pd.read_csv(OMOP / 'observation.csv',
                  usecols=['person_id', 'observation_concept_id', 'observation_date',
                           'value_as_number', 'value_as_concept_id', 'observation_source_value'],
                  dtype={'observation_source_value': str})
meas = pd.read_csv(OMOP / 'measurement.csv',
                   usecols=['person_id', 'measurement_concept_id', 'measurement_date',
                            'value_as_number', 'value_as_concept_id',
                            'value_source_value', 'measurement_source_value'],
                   dtype={'measurement_source_value': str, 'value_source_value': str})
cond = pd.read_csv(OMOP / 'condition_occurrence.csv',
                   usecols=['person_id', 'condition_source_value'], dtype=str)
cond['person_id'] = cond['person_id'].astype(int)
drug = pd.read_csv(OMOP / 'drug_exposure.csv', usecols=['person_id', 'drug_concept_id'])
death = pd.read_csv(OMOP / 'death.csv', usecols=['person_id'])

def bids(person_ids):
    """Map an iterable of person_ids to a set of BIDs."""
    return set(PID2BID.reindex(pd.unique(pd.Series(list(person_ids)))).dropna())

def omop_first(df, datecol, valcol='value_as_number'):
    """Earliest USABLE record per person -> BID-indexed value Series.

    Null values are dropped BEFORE picking the earliest row, so a subject
    whose first record has no value contributes their first record that
    does. Ascending date sort + drop_duplicates(keep='first') then keeps
    each person's earliest remaining (i.e., baseline/screening) row.
    """
    d = df.dropna(subset=[valcol]).sort_values(datecol).drop_duplicates('person_id')
    out = d.set_index('person_id')[valcol]
    out.index = PID2BID.reindex(out.index)
    return out[out.index.notna()]

def m_by_src(prefix, exact=False):
    if exact:
        return meas[meas.measurement_source_value == prefix]
    return meas[meas.measurement_source_value.str.startswith(prefix, na=False)]

print(f'OMOP loaded: {len(person):,} persons, {len(meas):,} measurements, '
      f'{len(obs):,} observations')

## Effort 1 — Population / cohort fidelity

Two batteries. **Battery A (trial populations)** are largely identity-mapped
through `BID` and serve as the sanity floor; **Battery B (phenotypes)** are
criteria-based definitions that exercise value mapping, unit conversion, QC
filtering, and domain routing — the places ETL drift would actually appear.
`A8 mITT` is a deliberate known-limitation probe: the ADaM `MITTFL` flag was
excluded from the CDM as analysis metadata, so the OMOP side must *reconstruct*
the modified intention-to-treat population (randomized + ≥2 PACC dates).

In [ ]:
# ── Source tables for cohort derivations ────────────────────────────
subj = pd.read_csv(BASE / 'Derived Data/SUBJINFO.csv', usecols=['BID', 'TX', 'SUBSTUDY'])
ds = pd.read_csv(BASE / 'Derived Data/DS.csv', usecols=['BID', 'DSDECOD'])
# ADQS is long format (one row per questionnaire record); these are
# subject-level flags repeated on every row, so collapse to one row per BID
adqs1 = (pd.read_csv(BASE / 'Derived Data/ADQS.csv',
                     usecols=['BID', 'MITTFL', 'APOEGNPRSNFLG'], low_memory=False)
         .groupby('BID').first())
amy = pd.read_csv(BASE / 'External Data/imaging_SUVR_amyloid.csv', low_memory=False)
tau_src = pd.read_csv(BASE / 'External Data/imaging_SUVR_tau.csv',
                      usecols=['BID', 'scan_analyzed'], low_memory=False)
psych = pd.read_csv(BASE / 'Raw Data/psychwell.csv', usecols=['BID', 'DONE', 'GDTOTAL'])
mmse_src = pd.read_csv(BASE / 'Raw Data/mmse.csv', usecols=['BID', 'DONE', 'MMSCORE'])
cdr_src = pd.read_csv(BASE / 'Raw Data/cdr.csv', usecols=['BID', 'DONE', 'CDSOB'])
phy = pd.read_csv(BASE / 'Raw Data/phyneuro.csv')
dose = pd.read_csv(BASE / 'Raw Data/dose.csv', usecols=['BID', 'DONE'])
fampar = pd.read_csv(BASE / 'Raw Data/famhxpar.csv', usecols=['BID', 'MOTHER', 'FATHER'])
famsib = pd.read_csv(BASE / 'Raw Data/famhxsib.csv', usecols=['BID', 'SIBDEMENT'])
habits = pd.read_csv(BASE / 'Raw Data/habits.csv', usecols=['BID', 'DONE', 'SMOKE'])

# Screening amyloid composite -> centiloid >= 20, natively per side
def src_amyloid_positive():
    a = amy[(amy.scan_analyzed == 'Yes') & (amy.brain_region == 'Composite_Summary')].copy()
    a['suvr'] = pd.to_numeric(a.suvr_cer, errors='coerce')
    a['days'] = pd.to_numeric(a.scan_date_DAYS_CONSENT, errors='coerce')
    # earliest analyzable scan per subject (= the screening scan): rows
    # without a SUVR can't win, undated rows sort last, dedup keeps first
    a = (a.dropna(subset=['suvr']).sort_values('days', na_position='last')
         .drop_duplicates('BID'))
    # AMYLCENT formula from the A4 data dictionary; >= 20 CL = trial positivity cut
    return set(a.loc[183.07 * a.suvr - 177.26 >= 20, 'BID'])

def omop_amyloid_positive():
    a = meas[(meas.measurement_concept_id == 2100000031)
             & meas.measurement_source_value.str.contains('Composite_Summary', na=False)]
    a = a.sort_values('measurement_date').drop_duplicates('person_id')
    return bids(a.loc[183.07 * a.value_as_number - 177.26 >= 20, 'person_id'])

EXAM_FIELDS = ['PXHEADEY', 'PXCARD', 'PXPULM', 'PXABDOM', 'PXMUSCUL', 'PXEDEMA',
               'PXSKIN', 'PXOTHER', 'NXGAIT', 'NXMOTOR', 'NXSENSOR', 'NXTREMOR',
               'NXFINGER', 'NXHEEL', 'NXNERVE', 'NXOTHER']

def src_abnormal_exam():
    # DONE is only recorded at follow-up; blank at screening means the exam
    # WAS done, so missing counts as done (same rule the ETL applies)
    p = phy[phy.DONE.fillna(1) == 1]
    return set(p.loc[(p[EXAM_FIELDS] == 2).any(axis=1), 'BID'])

def omop_mitt():
    """Reconstruction: randomized AND >= 2 distinct PACC dates (baseline + post-baseline)."""
    rand = set(obs.loc[obs.observation_concept_id == 618771, 'person_id'])
    n_dates = (meas[meas.measurement_concept_id == 2100000001]
               .groupby('person_id')['measurement_date'].nunique())
    return bids(rand & set(n_dates[n_dates >= 2].index))

YES = 4188539  # SNOMED "Yes"

In [ ]:
# ── Cohort battery: (name, source-derived set, OMOP-derived set, note) ──
cohorts = [
    ('A1 Enrolled (identity)', set(subj.BID), set(person.person_source_value), ''),
    ('A2 Randomized', set(subj.loc[subj.TX.notna(), 'BID']),
     bids(obs.loc[obs.observation_concept_id == 618771, 'person_id']), ''),
    ('A3 Solanezumab arm', set(subj.loc[subj.TX == 'Solanezumab', 'BID']),
     bids(obs.loc[(obs.observation_concept_id == 618771)
                  & (obs.value_as_concept_id == 36852349), 'person_id']), ''),
    ('A4 Placebo arm', set(subj.loc[subj.TX == 'Placebo', 'BID']),
     bids(obs.loc[(obs.observation_concept_id == 618771)
                  & (obs.value_as_concept_id == 44804245), 'person_id']), ''),
    ('A5 Study completed', set(ds.loc[ds.DSDECOD == 'COMPLETED', 'BID']),
     bids(obs.loc[obs.observation_concept_id == 40482840, 'person_id']), ''),
    ('A6 Screen failure', set(ds.loc[ds.DSDECOD == 'SCREEN FAILURE', 'BID']),
     bids(obs.loc[obs.observation_concept_id == 40480675, 'person_id']), ''),
    ('A7 Died', set(ds.loc[ds.DSDECOD == 'DEATH', 'BID']), bids(death.person_id), ''),
    ('A8 mITT (reconstructed)', set(adqs1.loc[adqs1.MITTFL == 1].index), omop_mitt(),
     'ADaM MITTFL is analysis metadata excluded from the CDM by design; '
     'OMOP side reconstructs it as randomized + >=2 PACC dates'),
    ('B1 Amyloid-positive (>=20 CL)', src_amyloid_positive(), omop_amyloid_positive(), ''),
    ('B2 APOE e4 carrier', set(adqs1.loc[adqs1.APOEGNPRSNFLG == 1].index),
     bids(meas.loc[(meas.measurement_concept_id == 3006041)
                   & (meas.value_as_concept_id == YES), 'person_id']), ''),
    ('B3 GDS > 5 (any visit)',
     set(psych.loc[(psych.DONE == 1) & (psych.GDTOTAL > 5), 'BID']),
     bids(obs.loc[(obs.observation_concept_id == 3051694)
                  & (obs.value_as_number > 5), 'person_id']), ''),
    ('B4 MMSE < 27 (any visit)',
     set(mmse_src.loc[(mmse_src.DONE == 'Yes') & (mmse_src.MMSCORE < 27), 'BID']),
     bids(obs.loc[(obs.observation_concept_id == 42869860)
                  & (obs.value_as_number < 27), 'person_id']), ''),
    ('B5 CDR-SB > 0 (any visit)',
     set(cdr_src.loc[(cdr_src.DONE == 'Yes') & (cdr_src.CDSOB > 0), 'BID']),
     bids(meas.loc[(meas.measurement_concept_id == 37524289)
                   & (meas.value_as_number > 0), 'person_id']), ''),
    ('B6 Abnormal exam finding (ever)', src_abnormal_exam(),
     bids(cond.loc[cond.condition_source_value.str.startswith('PHYNEURO', na=False),
                   'person_id']), ''),
    ('B7 Solanezumab-exposed',
     set(subj.loc[subj.TX == 'Solanezumab', 'BID']) & set(dose.loc[dose.DONE == 'Yes', 'BID']),
     bids(drug.loc[drug.drug_concept_id == 36852349, 'person_id']), ''),
    ('B8 Family hx dementia (parent/sib)',
     set(fampar.loc[(fampar.MOTHER == 1) | (fampar.FATHER == 1), 'BID'])
     | set(famsib.loc[famsib.SIBDEMENT == 1, 'BID']),
     bids(obs.loc[obs.observation_concept_id.isin([2100000210, 2100000211, 2100000212])
                  & (obs.value_as_concept_id == YES), 'person_id']), ''),
    ('B9 Tau-PET scanned', set(tau_src.loc[tau_src.scan_analyzed == 'Yes', 'BID']),
     bids(meas.loc[meas.measurement_source_value.str.startswith('TAU|', na=False),
                   'person_id']), ''),
    ('B10 Smoker >=1 pack/day (any visit)',
     set(habits.loc[(habits.DONE == 1) & (habits.SMOKE >= 1), 'BID']),
     bids(obs.loc[(obs.observation_concept_id == 35810373)
                  & (obs.value_as_number >= 20), 'person_id']),
     'Criterion expressed natively per side: >=1 pack/day (source) '
     'vs >=20 cigarettes/day (OMOP, x20 pack conversion)'),
]

rows = []
for name, s, o, note in cohorts:
    inter, union = s & o, s | o
    rows.append({'cohort': name, 'n_source': len(s), 'n_omop': len(o),
                 'n_intersection': len(inter), 'n_union': len(union),
                 'jaccard': len(inter) / len(union) if union else np.nan,
                 'source_only': len(s - o), 'omop_only': len(o - s), 'note': note})
cohort_res = pd.DataFrame(rows)
cohort_res.to_csv(RESULTS_DIR / 'etl_validation_cohorts.csv', index=False)
print(cohort_res.drop(columns='note').to_string(index=False,
      float_format=lambda v: f'{v:.4f}'))
for name, s, o, note in cohorts:
    if s != o:
        print(f'\nDiscordant — {name}: source_only={len(s-o)}, omop_only={len(o-s)}')
        if note:
            print(f'  {note}')

In [ ]:
# ── Figure: Jaccard by cohort ────────────────────────────────────────
d = cohort_res.iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(range(len(d)), d.jaccard, height=0.62, color=C_OMOP)
for i, (j, n) in enumerate(zip(d.jaccard, d.n_union)):
    ax.text(0.005, i, f'{j:.4f}  (n={n:,})', va='center', ha='left',
            color='white', fontsize=8.5, fontweight='bold')
ax.set_yticks(range(len(d)))
ax.set_yticklabels(d.cohort, fontsize=9)
ax.set_xlabel('Jaccard similarity  |S ∩ O| / |S ∪ O|')
ax.set_xlim(0, 1.0)
ax.set_title('Cohort fidelity: source-derived vs OMOP-derived cohorts')
ax.spines[['top', 'right']].set_visible(False)
ax.xaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'etl_validation_jaccard.png', dpi=150, bbox_inches='tight')
plt.show()

### Effort 1 interpretation

In the current build, **17 of 18 cohorts reproduce with Jaccard = 1.0000** —
identical participant sets from both derivation routes, including every
criteria-based phenotype (value thresholds, QC-filtered imaging, unit-converted
smoking, family-history value concepts, exam-finding conditions). The single
divergence is the deliberate probe: **A8 mITT** (Jaccard ≈ 0.98, OMOP-only
subjects) — the ADaM `MITTFL` flag encodes analysis decisions that are not fully
derivable from clinical data alone, which is precisely why analysis-population
flags were excluded from the CDM. Any downstream mITT analysis should treat the
reconstruction as an approximation.

## Effort 2 — Analytic fidelity (ML performance deviation)

Design: subjects present on both sides form the paired cohort; features, labels,
and preprocessing are computed *natively* per side; folds are assigned once by
`MD5(BID) mod 5` and reused on both sides; out-of-fold predictions are pooled to
a single AUROC per side; `Δ = AUROC_OMOP − AUROC_Source` gets a 95% CI from a
2,000-resample paired bootstrap over subjects. Model/CV specs match Analysis 5
exactly (median imputation, LR max_iter=1000 / RF 200×5 / GB 200×3 lr=0.1, all
`random_state=42`).

In [ ]:
# ── Task A: p-tau217 -> amyloid-PET positivity (mirrors Analysis 2) ──
ml_rows = []

# OMOP side: earliest p-tau217 (1092155); label from screening composite CL >= 20
ptau_o = omop_first(meas[meas.measurement_concept_id == 1092155], 'measurement_date')
amyc = meas[(meas.measurement_concept_id == 2100000031)
            & meas.measurement_source_value.str.contains('Composite_Summary', na=False)]
suvr_o = omop_first(amyc, 'measurement_date')
label_o = (183.07 * suvr_o - 177.26 >= 20).astype(int)

# Source side: numeric ORRES only (native pipeline: '<LLOQ' text coerces to NaN)
pt = pd.read_csv(BASE / 'External Data/biomarker_pTau217.csv', low_memory=False)
pt['v'] = pd.to_numeric(pt.ORRES, errors='coerce')
pt['d'] = pd.to_numeric(pt.COLLECTION_DATE_DAYS_CONSENT, errors='coerce')
# Earliest QUANTIFIED draw per subject: dropping NaN 'v' first means a
# censored (<LLOQ) first draw can never be selected; sorting by days-from-
# consent with undated draws last, then drop_duplicates(keep='first'),
# takes each subject's earliest row that still has a usable number.
ptau_s = (pt.dropna(subset=['v']).sort_values('d', na_position='last')
          .drop_duplicates('BID').set_index('BID')['v'])
# same earliest-usable-scan selection as Effort 1 (screening composite SUVR)
a = amy[(amy.scan_analyzed == 'Yes') & (amy.brain_region == 'Composite_Summary')].copy()
a['suvr'] = pd.to_numeric(a.suvr_cer, errors='coerce')
a['d'] = pd.to_numeric(a.scan_date_DAYS_CONSENT, errors='coerce')
suvr_s = (a.dropna(subset=['suvr']).sort_values('d', na_position='last')
          .drop_duplicates('BID').set_index('BID')['suvr'])
label_s = (183.07 * suvr_s - 177.26 >= 20).astype(int)

common_a = sorted(set(ptau_o.index) & set(label_o.index)
                  & set(ptau_s.index) & set(label_s.index))
xo, yo = ptau_o[common_a].values, label_o[common_a].values
xs, ys = ptau_s[common_a].values, label_s[common_a].values

def paired_delta_ci(y1, p1, y2, p2, n_boot=2000):
    idx = np.arange(len(y1))
    ds = []
    for _ in range(n_boot):
        b = rng.choice(idx, len(idx), replace=True)
        if len(set(y1[b])) < 2 or len(set(y2[b])) < 2:
            continue  # AUROC is undefined if a resample draws only one class
        ds.append(roc_auc_score(y1[b], p1[b]) - roc_auc_score(y2[b], p2[b]))
    return np.percentile(ds, [2.5, 97.5])

auc_o, auc_s = roc_auc_score(yo, xo), roc_auc_score(ys, xs)
lo, hi = paired_delta_ci(yo, xo, ys, xs)
ml_rows.append({'task': 'A: p-tau217 -> amyloid+ (AUROC)', 'n_paired': len(common_a),
                'auc_omop': auc_o, 'auc_source': auc_s, 'delta': auc_o - auc_s,
                'ci_lo': lo, 'ci_hi': hi})
n_src_full = len(set(ptau_s.index) & set(label_s.index))
n_omop_full = len(set(ptau_o.index) & set(label_o.index))
print(f'Task A cohorts: source n={n_src_full}, OMOP n={n_omop_full}, paired n={len(common_a)}')
print(f'  (OMOP-only subjects are the all-<LLOQ cases the ETL recovers via ORRESRAW)')
print(f'  label agreement on paired: {(yo == ys).mean():.4f}; '
      f'feature Pearson r = {np.corrcoef(xo, xs)[0, 1]:.5f}')
print(f'  AUROC: OMOP {auc_o:.4f} vs source {auc_s:.4f} '
      f'-> delta {auc_o - auc_s:+.4f} [95% CI {lo:+.4f}, {hi:+.4f}]')

In [ ]:
# ── Task B outcome per side: PACC decline > 1 SD (mirrors Analysis 5) ──
def outcome_omop():
    p = (meas[meas.measurement_source_value == 'PACC:PACC.raw']
         .dropna(subset=['value_as_number']).sort_values('measurement_date'))
    bl = p.drop_duplicates('person_id', keep='first').set_index('person_id')['value_as_number']
    last = p.drop_duplicates('person_id', keep='last').set_index('person_id')['value_as_number']
    # change-from-baseline needs a baseline AND at least one later assessment
    keep = p.groupby('person_id').size().pipe(lambda n: n[n >= 2].index)
    thr = -1 * bl.loc[keep].std()
    y = ((last - bl).loc[keep] < thr).astype(int)
    y.index = PID2BID.reindex(y.index)
    return y[y.index.notna()], thr

def outcome_source():
    p = pd.read_csv(BASE / 'Derived Data/PACC.csv', usecols=['BID', 'VISCODE', 'PACC.raw'])
    p = p.dropna(subset=['PACC.raw']).copy()
    # PACC.csv carries no date column; numeric visit codes are the
    # chronological order, so first = baseline, last = final assessment
    p['vis'] = pd.to_numeric(p.VISCODE, errors='coerce')
    p = p.sort_values('vis')
    bl = p.drop_duplicates('BID', keep='first').set_index('BID')['PACC.raw']
    last = p.drop_duplicates('BID', keep='last').set_index('BID')['PACC.raw']
    keep = p.groupby('BID').size().pipe(lambda n: n[n >= 2].index)
    thr = -1 * bl.loc[keep].std()
    return ((last - bl).loc[keep] < thr).astype(int), thr

y_omop, thr_o = outcome_omop()
y_src, thr_s = outcome_source()
common_b = sorted(set(y_omop.index) & set(y_src.index))
yo_b, ys_b = y_omop[common_b], y_src[common_b]
print(f'Outcome cohorts: source n={len(y_src)} (threshold {thr_s:.3f}), '
      f'OMOP n={len(y_omop)} (threshold {thr_o:.3f}), paired n={len(common_b)}')
print(f'Label concordance: agreement {(yo_b.values == ys_b.values).mean():.4f}, '
      f'kappa {cohen_kappa_score(yo_b, ys_b):.4f}; '
      f'prevalence source {ys_b.mean():.3f} / OMOP {yo_b.mean():.3f}')

In [ ]:
# ── Task B features per side (Analysis 5's 18-feature set, native values) ──
def features_omop(index):
    f = pd.DataFrame(index=index)
    pb = person.set_index('person_source_value')
    f['age'] = 2020 - pb.reindex(index)['year_of_birth']          # year-precision
    f['female'] = (pb.reindex(index)['gender_concept_id'] == 8532).astype(float)
    f['white'] = (pb.reindex(index)['race_concept_id'] == 8527).astype(float)
    f['mmse'] = omop_first(obs[obs.observation_concept_id == 42869860],
                           'observation_date').reindex(index)     # OBSERVATION post-routing
    f['cdr_sb'] = omop_first(m_by_src('CDR:CDSOB'), 'measurement_date').reindex(index)
    f['ptau217'] = ptau_o.reindex(index)                          # incl. <LLOQ recoveries
    f['amyloid_suvr'] = suvr_o.reindex(index)
    f['nfl'] = omop_first(m_by_src('ROCHE:NF-L'), 'measurement_date').reindex(index)
    f['gfap'] = omop_first(m_by_src('ROCHE:GFAP'), 'measurement_date').reindex(index)
    lh = omop_first(m_by_src('MRI:LeftHippocampus', exact=True), 'measurement_date')
    rh = omop_first(m_by_src('MRI:RightHippocampus', exact=True), 'measurement_date')
    f['hippocampal_vol'] = ((lh + rh) / 2).reindex(index)         # -4 sentinels filtered
    f['weight'] = omop_first(m_by_src('STDWT', exact=True), 'measurement_date').reindex(index)
    f['systolic_bp'] = omop_first(m_by_src('VSBPSYS', exact=True),
                                  'measurement_date').reindex(index)
    def ob(prefix):
        return omop_first(obs[obs.observation_source_value.str.startswith(prefix, na=False)],
                          'observation_date').reindex(index)
    f['smoking'] = ob('HABITS:SMOKE')                             # cigarettes/day (x20)
    f['alcohol'] = ob('HABITS:ALCOHOL')
    f['exercise'] = ob('HABITS:AEROBIC')
    f['family_hx_mother'] = ob('FAMHX:MOTHER')
    f['apoe_e4_carrier'] = omop_first(m_by_src('ADQS:APOEGNPRSNFLG'),
                                      'measurement_date').reindex(index)
    tx = obs[obs.observation_concept_id == 618771].copy()
    tx['bid'] = PID2BID.reindex(tx.person_id).values
    f['tx_solanezumab'] = (tx.set_index('bid')['value_as_concept_id']
                           .map({36852349: 1.0, 44804245: 0.0}).reindex(index))
    return f

def src_first(df, valcol, ordcol, filt=None):
    """Earliest usable value per BID from a source table (mirror of omop_first).

    to_numeric(errors='coerce') turns non-numeric results (e.g. '<LLOQ')
    into NaN, which dropna then removes — only quantified values compete.
    `ordcol` is a visit code or days-from-consent offset; both order
    chronologically, and rows without one sort last so a dated/coded row
    always beats an undated one. drop_duplicates keeps the earliest.
    """
    d = (df if filt is None else df[filt]).copy()
    d['v'] = pd.to_numeric(d[valcol], errors='coerce')
    d['o'] = pd.to_numeric(d[ordcol], errors='coerce')
    d = d.dropna(subset=['v']).sort_values('o', na_position='last').drop_duplicates('BID')
    return d.set_index('BID')['v']

def features_source(index):
    f = pd.DataFrame(index=index)
    s = (pd.read_csv(BASE / 'Derived Data/SUBJINFO.csv',
                     usecols=['BID', 'TX', 'AGEYR', 'SEX', 'RACE'])
         .set_index('BID').reindex(index))
    f['age'] = s['AGEYR']                                         # fractional age
    f['female'] = (s['SEX'] == 1).astype(float)
    f['white'] = (s['RACE'] == 1).astype(float)
    mm = pd.read_csv(BASE / 'Raw Data/mmse.csv', usecols=['BID', 'VISCODE', 'DONE', 'MMSCORE'])
    f['mmse'] = src_first(mm, 'MMSCORE', 'VISCODE', mm.DONE == 'Yes').reindex(index)
    cd = pd.read_csv(BASE / 'Raw Data/cdr.csv', usecols=['BID', 'VISCODE', 'DONE', 'CDSOB'])
    f['cdr_sb'] = src_first(cd, 'CDSOB', 'VISCODE', cd.DONE == 'Yes').reindex(index)
    f['ptau217'] = ptau_s.reindex(index)                          # numeric ORRES only
    f['amyloid_suvr'] = suvr_s.reindex(index)
    roche = pd.read_csv(BASE / 'External Data/biomarker_Plasma_Roche_Results.csv',
                        low_memory=False)
    roche['code'] = roche.LBTESTCD.astype(str).str.strip()  # source has 'NF-L ' variants
    f['nfl'] = src_first(roche, 'LABRESN', 'LABD_DAYS_CONSENT',
                         roche.code.isin(['NF-L', 'NFL'])).reindex(index)
    f['gfap'] = src_first(roche, 'LABRESN', 'LABD_DAYS_CONSENT',
                          roche.code == 'GFAP').reindex(index)
    mri = pd.read_csv(BASE / 'External Data/imaging_volumetric_mri.csv', low_memory=False)
    lh = src_first(mri, 'LeftHippocampus', 'Date_DAYS_CONSENT')
    rh = src_first(mri, 'RightHippocampus', 'Date_DAYS_CONSENT')
    f['hippocampal_vol'] = ((lh + rh) / 2).reindex(index)         # -4 sentinels flow through
    vit = pd.read_csv(BASE / 'Raw Data/vitals.csv',
                      usecols=['BID', 'VISCODE', 'DONE', 'STDWT', 'VSBPSYS'])
    f['weight'] = src_first(vit, 'STDWT', 'VISCODE', vit.DONE == 1).reindex(index)
    f['systolic_bp'] = src_first(vit, 'VSBPSYS', 'VISCODE', vit.DONE == 1).reindex(index)
    hab = pd.read_csv(BASE / 'Raw Data/habits.csv',
                      usecols=['BID', 'VISCODE', 'DONE', 'SMOKE', 'ALCOHOL', 'AEROBIC'])
    for col, name in [('SMOKE', 'smoking'), ('ALCOHOL', 'alcohol'), ('AEROBIC', 'exercise')]:
        f[name] = src_first(hab, col, 'VISCODE', hab.DONE == 1).reindex(index)  # packs/day
    fp = pd.read_csv(BASE / 'Raw Data/famhxpar.csv', usecols=['BID', 'VISCODE', 'MOTHER'])
    f['family_hx_mother'] = src_first(fp, 'MOTHER', 'VISCODE').reindex(index)
    f['apoe_e4_carrier'] = (pd.read_csv(BASE / 'Derived Data/ADQS.csv',
                                        usecols=['BID', 'APOEGNPRSNFLG'], low_memory=False)
                            .groupby('BID')['APOEGNPRSNFLG'].first().reindex(index))
    f['tx_solanezumab'] = s['TX'].map({'Solanezumab': 1.0, 'Placebo': 0.0})
    return f

X_omop = features_omop(pd.Index(common_b))
X_src = features_source(pd.Index(common_b))

conc = []
for c in X_omop.columns:
    both = X_omop[c].notna() & X_src[c].notna()
    r = (np.corrcoef(X_omop.loc[both, c], X_src.loc[both, c])[0, 1]
         if both.sum() > 2 else np.nan)
    conc.append({'feature': c, 'n_omop': int(X_omop[c].notna().sum()),
                 'n_source': int(X_src[c].notna().sum()), 'n_both': int(both.sum()),
                 'pearson_r': r})
conc = pd.DataFrame(conc)
conc.to_csv(RESULTS_DIR / 'etl_validation_feature_concordance.csv', index=False)
print('Feature concordance on the paired cohort:')
print(conc.to_string(index=False, float_format=lambda v: f'{v:.5f}'))

# Analysis 5's >=30%-availability screen, applied JOINTLY: a per-side rule
# could keep different feature sets and the models would no longer be comparable
valid = [c for c in X_omop.columns
         if X_omop[c].notna().mean() >= 0.30 and X_src[c].notna().mean() >= 0.30]
print(f'\nFeatures kept (>=30% availability on both sides): {len(valid)} of '
      f'{len(X_omop.columns)}')

In [ ]:
# ── Task B: identical models, identical folds, native data per side ──
models = {
    'Logistic Regression': lambda: Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'Random Forest': lambda: Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42))]),
    'Gradient Boosting': lambda: Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                           learning_rate=0.1, random_state=42))]),
}
# Fold membership is a pure function of the subject ID: deterministic and
# identical on both sides. sklearn's KFold would need a shared permutation,
# and StratifiedKFold could split differently where the two sides' labels
# disagree — hashing sidesteps both.
folds = np.array([int(hashlib.md5(b.encode()).hexdigest(), 16) % 5 for b in common_b])

def oof_probs(X, y, make_model):
    """Out-of-fold probabilities under the shared BID-hash fold assignment."""
    p = np.full(len(y), np.nan)
    for k in range(5):
        m = make_model()
        m.fit(X[folds != k], y[folds != k])
        p[folds == k] = m.predict_proba(X[folds == k])[:, 1]
    return p

print('Paired 5-fold OOF AUROC (identical folds; native labels/features per side):')
for name, make in models.items():
    po = oof_probs(X_omop[valid].values, yo_b.values, make)
    ps = oof_probs(X_src[valid].values, ys_b.values, make)
    ao, as_ = roc_auc_score(yo_b, po), roc_auc_score(ys_b, ps)
    lo, hi = paired_delta_ci(yo_b.values, po, ys_b.values, ps)
    fold_d = [roc_auc_score(yo_b.values[folds == k], po[folds == k])
              - roc_auc_score(ys_b.values[folds == k], ps[folds == k]) for k in range(5)]
    ml_rows.append({'task': f'B: PACC decline, {name}', 'n_paired': len(common_b),
                    'auc_omop': ao, 'auc_source': as_, 'delta': ao - as_,
                    'ci_lo': lo, 'ci_hi': hi})
    print(f'  {name:22s} OMOP {ao:.4f} vs source {as_:.4f} '
          f'-> delta {ao - as_:+.4f} [95% CI {lo:+.4f}, {hi:+.4f}]  '
          f'per-fold: {[f"{d:+.3f}" for d in fold_d]}')

ml_res = pd.DataFrame(ml_rows)
ml_res.to_csv(RESULTS_DIR / 'etl_validation_ml.csv', index=False)
ml_res

In [ ]:
# ── Figure: AUROC by side, with paired-bootstrap delta CI ────────────
# Dot marks (not bars): AUROC lives on a truncated 0.5-1.0 scale, where bar
# lengths would mislead; points with a marked chance floor do not.
labels = ['Task A\np-tau217 -> amyloid+', 'Task B\nLogistic Regression',
          'Task B\nRandom Forest', 'Task B\nGradient Boosting']
x = np.arange(len(ml_res))
off = 0.13
fig, ax = plt.subplots(figsize=(9, 5))
for i, row in ml_res.iterrows():
    ax.plot([i - off, i + off], [row.auc_source, row.auc_omop],
            color='#B0B6BB', lw=1.2, zorder=1)
ax.scatter(x - off, ml_res.auc_source, s=90, color=C_SOURCE, zorder=3,
           label='Source-derived', edgecolors='white', linewidths=1.5)
ax.scatter(x + off, ml_res.auc_omop, s=90, color=C_OMOP, zorder=3,
           label='OMOP-derived', edgecolors='white', linewidths=1.5)
for i, row in ml_res.iterrows():
    ax.text(i - off, row.auc_source - 0.016, f'{row.auc_source:.3f}',
            ha='center', va='top', fontsize=8.5, color='#444444')
    ax.text(i + off, row.auc_omop + 0.016, f'{row.auc_omop:.3f}',
            ha='center', va='bottom', fontsize=8.5, color='#444444')
    ax.text(i, 0.565, f'Δ {row.delta:+.4f}\n[{row.ci_lo:+.3f}, {row.ci_hi:+.3f}]',
            ha='center', fontsize=8, color='#444444')
ax.axhline(0.5, color='#999999', lw=1, ls='--')
ax.text(len(x) - 0.52, 0.503, 'chance', fontsize=8, color='#777777', ha='right')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('AUROC (pooled out-of-fold)')
ax.set_ylim(0.49, 0.92)
ax.set_xlim(-0.6, len(x) - 0.4)
ax.set_title('Analytic fidelity: identical pipeline on source vs OMOP data')
ax.legend(frameon=False, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'etl_validation_auroc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Deviation attribution: trace every sub-unity concordance to its cause ──
# 1) hippocampal_vol: source-side -4 missing-value sentinels (OMOP filters them)
mri = pd.read_csv(BASE / 'External Data/imaging_volumetric_mri.csv', low_memory=False)
mri['lh'] = pd.to_numeric(mri.LeftHippocampus, errors='coerce')
mri['d'] = pd.to_numeric(mri.Date_DAYS_CONSENT, errors='coerce')
# earliest scan with ANY LeftHippocampus value — including -4 sentinels,
# since the point is to count subjects whose earliest source row is one
first = (mri.dropna(subset=['lh']).sort_values('d', na_position='last')
         .drop_duplicates('BID'))
n_sentinel = int((first.lh < 0).sum())
both = X_omop.hippocampal_vol.notna() & X_src.hippocampal_vol.notna()
clean = both & (X_src.hippocampal_vol > 0)
r_all = np.corrcoef(X_omop.loc[both, 'hippocampal_vol'],
                    X_src.loc[both, 'hippocampal_vol'])[0, 1]
r_clean = np.corrcoef(X_omop.loc[clean, 'hippocampal_vol'],
                      X_src.loc[clean, 'hippocampal_vol'])[0, 1]
print(f'hippocampal_vol: r={r_all:.5f} overall; {n_sentinel} subjects whose earliest '
      f'source scan is the -4 sentinel row; excluding them r={r_clean:.5f}')

# 2) ptau217: <LLOQ recovery (ETL stores the raw instrument value with a "<" operator)
po = (meas[meas.measurement_concept_id == 1092155]
      .sort_values('measurement_date').drop_duplicates('person_id'))
po = po.assign(BID=PID2BID.reindex(po.person_id).values).set_index('BID')
common_pt = po.index.intersection(ptau_s.index)
diff = common_pt[(po.loc[common_pt, 'value_as_number'] - ptau_s.loc[common_pt]).abs() > 1e-9]
n_lloq = int(po.loc[diff, 'value_source_value'].astype(str).str.startswith('<').sum())
print(f'ptau217: {len(diff)} paired-value differences, {n_lloq} of them <LLOQ-recovered '
      f'rows; {len(set(po.index) - set(ptau_s.index))} OMOP-only subjects '
      f'(all-<LLOQ in source, unrecoverable natively)')

# 3) age: OMOP stores year_of_birth only
d_age = (X_omop.age - X_src.age).abs()
print(f'age: year-precision vs fractional AGEYR; max |diff| = {d_age.max():.2f} yr, '
      f'mean |diff| = {d_age.mean():.2f} yr')

### Effort 2 interpretation

In the current build, **no model shows a statistically distinguishable performance
deviation**: every `Δ = AUROC_OMOP − AUROC_Source` confidence interval spans zero
(Task A ≈ +0.005; Task B ≈ +0.002 / −0.002 / −0.008 for LR / RF / GB). Outcome
labels agree for 99.9–100% of paired subjects (κ ≈ 0.998), and 15 of 18 features
are numerically identical (r = 1.0000) across representations.

The three sub-unity concordances each trace to a **documented, intentional ETL
transform**, not to drift:

| Feature | r | Cause |
|---|---|---|
| `ptau217` | ≈ 0.998 | ETL recovers `<LLOQ` raw instrument values (with a `<` operator concept); the native source pipeline coerces the text to NaN — the OMOP side therefore has *more* usable subjects, and slightly different earliest values where a censored draw preceded a quantified one |
| `hippocampal_vol` | ≈ 0.77 | the source file's −4 missing-value sentinels flow through a native analysis as gross outliers; the ETL filters them — excluding those few subjects restores r ≈ 1.0 |
| `age` | ≈ 0.998 | OMOP stores `year_of_birth` only (year precision) vs fractional source age |

### Overall verdict

Under participant-level cohort comparison and a paired same-pipeline ML design,
the OMOP representation is **population-faithful** (Jaccard 1.0 on every cohort
derivable from clinical data; the one sub-1.0 cohort requires excluded analysis
metadata by construction) and **analytically faithful** (no significant AUROC
deviation on any task/model). Where the representations differ at the value
level, the differences are the ETL's documented data-quality improvements —
censored-value recovery and sentinel filtering — which act in the OMOP side's
favor rather than degrading it.